# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Dump metadata as dict for pretty print
metadata_dict = dataset.metadata.to_json()
print(f"{metadata_dict['name']}: {metadata_dict['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets with their @id and name
print("Available Record Sets in this Dataset:")
if not hasattr(dataset.metadata, 'record_sets') or not dataset.metadata.record_sets:
    print("No structured Record Sets found in metadata.")
else:
    for rs in dataset.metadata.record_sets:
        print(f"@id: {rs.id}, name: {rs.name if hasattr(rs, 'name') else '-'}")

### Explore fields for the main record set
Below, we print the available fields (columns) for each record set. If record sets are not available, we attempt to infer the main tabular distribution and print its fields.

In [ ]:
# Try to explore fields in record sets
if hasattr(dataset.metadata, 'record_sets') and dataset.metadata.record_sets:
    for rs in dataset.metadata.record_sets:
        print(f"\nRecordSet @id: {rs.id}")
        if hasattr(rs, 'fields'):
            for field in rs.fields:
                print(f"  Field @id: {field.id}, name: {getattr(field, 'name', '-')}, type: {getattr(field, 'data_type', '-')} ")
        elif hasattr(rs, 'columns'):
            for col in rs.columns:
                print(f"  Column @id: {col.id}, name: {getattr(col, 'name', '-')} ")
        else:
            print("  (No fields/columns defined)")
else:
    # If no record sets: try to infer from distributions (single main data file)
    print("No explicit record sets in metadata, attempting to infer main data table columns...")
    if hasattr(dataset.metadata, 'distributions'):
        for dist in dataset.metadata.distributions:
            # Try to print columns or fields from distribution
            if hasattr(dist, 'columns'):
                for col in dist.columns:
                    print(f"  Column @id: {col.id}, name: {getattr(col, 'name', '-')} ")
            else:
                print(f"  Distribution @id: {dist.id}, no columns defined")

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. We use the record set and field `@id` values as discovered above.

This dataset uses a single tabular data file without nested recordSet definitions, so we will use the default automatic record set. We'll extract all rows and display the columns.

In [ ]:
# If no record sets, use default: pass no record_set param for the main table.
records = list(dataset.records())  # This will load all rows from the main tabular distribution
df = pd.DataFrame(records)
print("Columns available in DataFrame:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

You can select fields from the column list above. For example, we analyze the 'Coverage (%)' field (if present), filtering records with high coverage and normalizing its values. Grouping is demonstrated using the 'Description' field if available.

In [ ]:
# Choose appropriate numeric and group fields by their column names (@id)
numeric_field = None
group_field = None

# Try common plausible column names
import re
for col in df.columns:
    if re.match(r"coverage", col, re.I) or "coverage" in col.lower():
        numeric_field = col
    if re.match(r"desc", col, re.I) or "description" in col.lower():
        group_field = col
if numeric_field is None:
    # Fallback to some other plausible quantitative field
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
if group_field is None:
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]) and "name" in col.lower():
            group_field = col
            break
print(f"Using numeric field: {numeric_field}")
print(f"Using group field: {group_field}")

# Ensure the numeric field is float for proper filtering/normalizing
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

threshold = 10
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold} (first five):")
print(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (
    (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
)
print(f"Normalized {numeric_field} for filtered records (first five):")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field}: (first five)")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We will plot a histogram of the main numeric field and, if a group field is available, a boxplot of the numeric field grouped by that field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Boxplot by group_field if available
if group_field is not None:
    plt.figure(figsize=(12, 5))
    # If too many unique groups, select top 10
    top_groups = filtered_df[group_field].value_counts().nlargest(10).index
    sns.boxplot(
        x=group_field, y=numeric_field, data=filtered_df[filtered_df[group_field].isin(top_groups)]
    )
    plt.title(f"{numeric_field} by {group_field} (Top 10 Groups)")
    plt.xticks(rotation=45, ha='right')
    plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load, explore, and process a Croissant-structured dataset of human protein mass spectrometry analysis using the `mlcroissant` library.

Key findings:
- The dataset contains protein abundance and descriptor fields suitable for EDA, such as coverage and description.
- Data can be filtered, normalized, and grouped for further study, enabling quantitative and categorical analysis.
- The Croissant format allows programmatic access to field-level metadata and machine-actionable workflows.

This approach can be extended to other Croissant-format datasets for interoperable ML data pipelines.